## Install Required Libraries

In [1]:
!pip install flask pyngrok -q
!pip install flask-cors -q


In [ ]:
import threading
from flask import Flask, request, jsonify, send_file
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.datasets import imdb
import numpy as np
from flask_cors import CORS
from pyngrok import ngrok


## Upload Trained Models

In [3]:
from google.colab import files
uploaded = files.upload()

Saving simple_rnn_model.h5 to simple_rnn_model.h5
Saving lstm_model.h5 to lstm_model.h5


## Build Flask API Server

In [4]:
app = Flask(__name__)
CORS(app)

rnn_model  = load_model('simple_rnn_model.h5')
lstm_model = load_model('lstm_model.h5')

word_index = imdb.get_word_index()

def text_to_sequence(text, max_len=200, max_features=10000):
    words = text.lower().split()
    sequence = []
    for word in words:
        idx = word_index.get(word, 2)
        if idx < max_features:
            sequence.append(idx + 3)
        else:
            sequence.append(2)
    padded = pad_sequences([sequence], maxlen=max_len)
    return padded

@app.route('/')
def index():
    return send_file('/content/index.html')

@app.after_request
def add_ngrok_header(response):
    response.headers['ngrok-skip-browser-warning'] = 'true'
    return response

@app.route('/predict', methods=['POST'])
def predict():
    data = request.get_json()
    text = data.get('text', '')

    if not text.strip():
        return jsonify({"error": "Mətn boşdur!"}), 400

    seq = text_to_sequence(text)

    rnn_score  = float(rnn_model.predict(seq, verbose=0)[0][0])
    lstm_score = float(lstm_model.predict(seq, verbose=0)[0][0])

    return jsonify({
        "rnn_result":  {"score": round(rnn_score, 4),
                        "label": "Positive" if rnn_score >= 0.5 else "Negative"},
        "lstm_result": {"score": round(lstm_score, 4),
                        "label": "Positive" if lstm_score >= 0.5 else "Negative"}
    })

def run_flask():
    app.run(port=5000)

thread = threading.Thread(target=run_flask)
thread.daemon = True
thread.start()
print("✅ Flask server işə düşdü!")

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
✅ Flask server işə düşdü!


## Launch Ngrok Tunnel

In [9]:

ngrok.set_auth_token("3A3snZIwZbjrNafN8EwhLmhJJ6p_5KLQoAjs32fUx9QaLT2kU")

ngrok.kill()
tunnel = ngrok.connect(5000)
public_url = tunnel.public_url
print(f"✅ URL: {public_url}")




✅ URL: https://seth-glariest-abbie.ngrok-free.dev


## Test the API

In [10]:
import requests

response = requests.post(
    'http://localhost:5000/predict',
    json={'text': 'this movie was great'}
)
print("Status:", response.status_code)
print("Nəticə:", response.json())



INFO:werkzeug:127.0.0.1 - - [28/Feb/2026 16:51:35] "POST /predict HTTP/1.1" 200 -


Status: 200
Nəticə: {'lstm_result': {'label': 'Negative', 'score': 0.4986}, 'rnn_result': {'label': 'Negative', 'score': 0.4289}}


## Create the Frontend (index.html)

In [11]:
api = str(public_url).strip()

with open('/content/index.html', 'w') as f:
    f.write('<!DOCTYPE html>\n')
    f.write('<html>\n<head><meta charset="UTF-8"><title>Sentiment</title>\n')
    f.write('<style>\n')
    f.write('body{font-family:Arial;background:#0f0f1a;color:#eee;text-align:center;padding:40px}\n')
    f.write('textarea{width:60%;height:120px;padding:12px;background:#1a1a2e;color:#eee;border:1px solid #444;border-radius:8px;font-size:1rem}\n')
    f.write('button{margin:16px auto;display:block;padding:12px 40px;background:#7b5ea7;color:#fff;border:none;border-radius:8px;font-size:1rem;cursor:pointer}\n')
    f.write('.results{display:flex;justify-content:center;gap:40px;margin-top:30px}\n')
    f.write('.card{background:#1a1a2e;border-radius:12px;padding:30px;width:220px;border:2px solid #333}\n')
    f.write('.positive{border-color:#4caf50}.negative{border-color:#f44336}\n')
    f.write('.score{font-size:2rem;font-weight:bold;margin:12px 0}\n')
    f.write('.positive .score{color:#4caf50}.negative .score{color:#f44336}\n')
    f.write('</style></head>\n<body>\n')
    f.write('<h1>🎬 RNN vs LSTM Sentiment Analyzer</h1>\n')
    f.write('<textarea id="txt" placeholder="Write a movie review..."></textarea>\n')
    f.write('<button id="btn">🔍 Analyze Sentiment</button>\n')
    f.write('<div class="results">\n')
    f.write('<div class="card" id="rnnCard"><h2>SimpleRNN</h2><div class="score" id="rnnScore">--</div><div id="rnnLabel">Waiting</div></div>\n')
    f.write('<div class="card" id="lstmCard"><h2>LSTM</h2><div class="score" id="lstmScore">--</div><div id="lstmLabel">Waiting</div></div>\n')
    f.write('</div>\n')
    f.write('<script>\n')
    f.write('var API = "' + api + '";\n')
    f.write('document.getElementById("btn").addEventListener("click", function() {\n')
    f.write('    var text = document.getElementById("txt").value.trim();\n')
    f.write('    if (!text) { alert("Please enter a review!"); return; }\n')
    f.write('    fetch(API + "/predict", {\n')
    f.write('        method: "POST",\n')
    f.write('        headers: {\n')
    f.write('            "Content-Type": "application/json",\n')
    f.write('            "ngrok-skip-browser-warning": "true"\n')  # ← əlavə edildi
    f.write('        },\n')
    f.write('        body: JSON.stringify({text: text})\n')
    f.write('    })\n')
    f.write('    .then(function(r) { return r.json(); })\n')
    f.write('    .then(function(data) {\n')
    f.write('        showResult("rnn", data.rnn_result);\n')
    f.write('        showResult("lstm", data.lstm_result);\n')
    f.write('    })\n')
    f.write('    .catch(function(e) { alert("Error: " + e.message); });\n')
    f.write('});\n')
    f.write('function showResult(model, result) {\n')
    f.write('    var isPos = result.label === "Positive";\n')
    f.write('    document.getElementById(model+"Card").className = "card "+(isPos?"positive":"negative");\n')
    f.write('    document.getElementById(model+"Score").textContent = (result.score*100).toFixed(1)+"%";\n')
    f.write('    document.getElementById(model+"Label").textContent = isPos ? "😊 Positive" : "😞 Negative";\n')
    f.write('}\n')
    f.write('</script>\n</body>\n</html>')

print("✅ index.html yaradıldı!")


✅ index.html yaradıldı!
